In [ ]:
#| default_exp vae.core

# core

> This module contains the implementation of the flax VAE model, losses and the trainer wrapper

In [1]:
#| hide
from nbdev.showdoc import *

In [2]:
#| export
#| hide

import jax
from jax import numpy as jnp
from flax import linen as nn

## VAE + losses

In [3]:
#| export
#| hide

#code inspired from the one found in https://github.com/YannDubs/disentangling-vae/tree/master

def matrix_log_density_gaussian(x: jnp.ndarray, mu: jnp.ndarray, logvar: jnp.ndarray) -> jnp.ndarray:
    """Calculates log density of a Gaussian for all batch pairs."""
    batch_size, dim = x.shape
    x = x[:, None, :]
    mu = mu[None, :, :]
    logvar = logvar[None, :, :]
    return log_density_gaussian(x, mu, logvar)

def log_importance_weight_matrix(batch_size: int, dataset_size: jnp.ndarray) -> jnp.ndarray:
    """Calculates a log importance weight matrix."""
    N = dataset_size
    M = batch_size - 1
    strat_weight = (N - M) / (N * M)
    W = jnp.full((batch_size, batch_size), 1 / M)
    W = W.at[jnp.diag_indices(batch_size)].set(1 / N)
    W = W.at[jnp.arange(1, batch_size), jnp.arange(batch_size - 1)].set(strat_weight)
    W = W.at[M - 1, 0].set(strat_weight)
    return jnp.log(W)

def _get_log_pz_qz_prodzi_qzCx(latent_sample: jnp.ndarray, latent_dist: jnp.ndarray, n_data: int, is_mss=True) -> jnp.ndarray:
    """compute log_pz, log_qz, log_prod_qzi, log_q_zCx"""
    batch_size, hidden_dim = latent_sample.shape

    # calculate log q(z|x)
    log_q_zCx = log_density_gaussian(latent_sample, *latent_dist).sum(axis=1)

    # calculate log p(z) (standard normal prior)
    zeros = jnp.zeros_like(latent_sample)
    log_pz = log_density_gaussian(latent_sample, zeros, zeros).sum(axis=1)

    mat_log_qz = matrix_log_density_gaussian(latent_sample, *latent_dist)

    if is_mss:
        # use stratification
        log_iw_mat = log_importance_weight_matrix(batch_size, n_data)
        mat_log_qz = mat_log_qz + log_iw_mat[:, :, None]

    log_qz = jax.scipy.special.logsumexp(mat_log_qz.sum(axis=2), axis=1)
    log_prod_qzi = jax.scipy.special.logsumexp(mat_log_qz, axis=1).sum(axis=1)

    return log_pz, log_qz, log_prod_qzi, log_q_zCx

In [4]:
#| export
#| hide

#from dataclasses import dataclass
#from typing import Any, Callable, Optional, Tuple, Dict
#import jax
#import jax.numpy as jnp
#import optax
#from flax import linen as nn
#from flax.training import train_state

class VAEmodel(nn.Module):
    """
        VAE model, wrapper calling the encoder -> reparam -> decoder
        
        Args:
        
            encoder: nn.Module
            decoder: nn.Module
            
        Returns:
        
            mean: (batch, latent_dim)
            logvar: (batch, latent_dim)
            z: (batch, latent_dim)
            cp: (batch, seq_len, local_dimension)
    """
    encoder: nn.Module
    decoder: nn.Module

    def reparameterize(self, mean, logvar, key):
        # Reparameterization trick
        std = jnp.exp(0.5 * logvar)
        eps = jax.random.normal(key, mean.shape)
        return mean + eps * std

    def __call__(self, batch, key):
        x = batch
        mean, logvar = self.encoder(x)
        key = jax.random.split(key)[0]
        z = self.reparameterize(mean, logvar, key)
        inputs = (z, x)
        cp = self.decoder(inputs)

        return mean, logvar, z, cp


    def get_latent(self, params, batch):
        key = jax.random.PRNGKey(0) #not used
        mean, logvar, _, _  = self.VAE.apply(params, batch, key)

        return mean, logvar


    def get_cprob(self, params, batch, key):
        _, _, _, cp = self.VAE.apply(params, batch, key)

        return cp


# Loss functions


def categorical_reconstruction_loss(log_cp_i: jnp.ndarray, targets: jnp.ndarray) -> jnp.ndarray:
    """ reconstruction loss for discrete data"""
    #nll = -jnp.take_along_axis(log_cp, targets[..., None].astype(jnp.int32), axis=-1).squeeze(-1)  # (B, seq_len)
    num_classes = jnp.shape(log_cp_i)[-1]
    log_cp = nn.one_hot(targets, num_classes)*log_cp_i
    return -jnp.sum(log_cp)/jnp.shape(targets)[0]


def gaussian_negloglik(recon_mean: jnp.ndarray, recon_logvar: jnp.ndarray, targets: jnp.ndarray, eps: float = 1e-8) -> jnp.ndarray:
    """ reconstruction loss for continuous data """
    var = jnp.exp(recon_logvar) + eps
    nll_per_dim = 0.5 * (jnp.log(2 * jnp.pi) + recon_logvar + ((targets - recon_mean) ** 2) / var)
    nll = jnp.sum(nll_per_dim, axis=tuple(range(1, nll_per_dim.ndim)))  # sum over non-batch dims
    return jnp.mean(nll)


def kl_standard_normal(mean: jnp.ndarray, logvar: jnp.ndarray) -> jnp.ndarray:
    """KL divergence between diag Gaussian q(z|x) ~ N(mean, exp(logvar)) and p(z)=N(0,I)."""
    kl = -0.5 * jnp.sum(1 + logvar - jnp.square(mean) - jnp.exp(logvar))
    return kl/jnp.shape(mean)[0]  # average over batch, sum over latent dims



def log_normal_pdf(mu: jnp.ndarray, logvar: jnp.ndarray, sample: jnp.ndarray):
    return -0.5 * ((sample - mu) ** 2 * jnp.exp(-logvar) + logvar + jnp.log(2. * jnp.pi))


def log_density_gaussian(x: jnp.ndarray, mu: jnp.ndarray, logvar: jnp.ndarray):
    """Calculates log density of a Gaussian."""
    normalization = -0.5 * (jnp.log(2 * jnp.pi) + logvar)
    inv_var = jnp.exp(-logvar)
    log_density = normalization - 0.5 * ((x - mu) ** 2 * inv_var)
    return log_density


#@jax.jit
def TC_term(mean: jnp.ndarray, logvar: jnp.ndarray, z: jnp.ndarray) -> jnp.ndarray:
    """compute TC term"""

    #KL loss
    log_pz, log_qz, log_prod_qzi, log_q_zCx  = _get_log_pz_qz_prodzi_qzCx(latent_sample=z, latent_dist=(mean,logvar), n_data=jnp.shape(z)[0], is_mss=True)

    #mi_loss = (log_q_zCx - log_qz).mean()
    tc_loss = (log_qz - log_prod_qzi).mean()
    #dw_kl_loss = (log_prod_qzi - log_pz).mean()
    return tc_loss


In [8]:
from qdisc.nn import EncoderCNN2D, ARNNDense

x = jnp.ones((100,10))
N = x.shape[-1]

encoder = EncoderCNN2D(latent_dim=5, conv_features = 4*N, dense_features = 4*N, num_conv_layers=3)
decoder = ARNNDense(num_layers=4, features=4*N)
myvae = VAEmodel(encoder=encoder, decoder=decoder)

key_init = jax.random.PRNGKey(3)
key_reparam = jax.random.PRNGKey(34)
params = myvae.init(key_init, x, key_reparam)

_, _, _, _ = myvae.apply(params, x, key_reparam)

# export

In [ ]:

#| hide
import nbdev; nbdev.nbdev_export()